In [1]:
import numpy as np
import pandas as pd
from itertools import combinations

In [2]:
def build_strategy_set(players):
    player_strategies = []
    player_index = []
    for pl in players:
        player_strategies.append(pl.strategy)
        player_index.append(pl.id)

    grids = np.meshgrid(*player_strategies)
    cartesian = np.column_stack([np.ravel(grid) for grid in grids])
    return pd.DataFrame(cartesian, columns=player_index)

def utility(s_i, s_minus_i, player_id, utilities):
    index_labels = s_minus_i.index.tolist()
    query = ""
    for pl_id in index_labels:
        query += f"`{pl_id}` == '{s_minus_i[pl_id]}' &"
    query += f" `{player_id}` == '{s_i}'"
    
    row = utilities.query(query)
    return row[f"payoff_{player_id}"].iloc[0]

def input_strategy_belief(input_text, available_belief):
    while True:
        probability = float(input(f"Probability for {input_text}: "))
        if probability <= available_belief:
            break
        else:
            print(f"Invalid Belief, Available Belief: {available_belief}. Try Again!!")
    return probability

def input_payoff(input_text, num_players):
    while True:
        payoff = input(input_text).split(" ")
        payoff = np.array(payoff, dtype=float)
        if len(payoff) == num_players:
            break
        else:
            print(f"Enter payoffs for {num_players} players.")
    return payoff

In [3]:
class Player:
    def __init__(self, id, strat):
        self.id = id
        self.strategy = np.array(strat, dtype=str)
        self.strategy_set_excluded = pd.DataFrame({})
        self.belief = pd.DataFrame({})
    
    def build_correlated_beliefs(self):
        beliefs = []
        available_belief = 1
        for _, strategy_profile in self.strategy_set_excluded.iterrows():
            probability = input_strategy_belief(strategy_profile, available_belief)            
            available_belief -= probability
            beliefs.append(probability)

        if sum(beliefs) != 1:
            print(f"Invalid Correlated Beliefs Built. Sum: {sum(beliefs)}. Try Again!!")
            self.build_correlated_beliefs(self)

        self.belief = pd.DataFrame(beliefs)
        self.strategy_set_excluded = pd.concat([self.strategy_set_excluded, self.belief], axis=1)

    def build_independent_beliefs(self):
        strat_probability = []
        for strat in self.strategy:
            strat_probability.append(input_strategy_belief(strat, 1))
        
        self.belief = pd.DataFrame({
            "strategy": self.strategy,
            "probability": strat_probability
        })
        # TODO: To construct belief of S_minus_i, we need to multiply individual players beliefs.
        # #We cannot construct this at player level because u(a,b,c) is dependent on other 
        # player's belief about a, b, c i.e. at game level, not player level.

    def find_dominated_strategies(self, utilities):
        dominated_strategies = [] # [(a, b)] a dominates b
        for s_i, s_j in list(combinations(self.strategy, 2)):
            i_minus_j = []
            for _, s_minus_i in self.strategy_set_excluded.iterrows():
                u_s_i = utility(s_i, s_minus_i, self.id, utilities)
                u_s_j = utility(s_j, s_minus_i, self.id, utilities)
                
                i_minus_j.append(u_s_i - u_s_j)
            
            if np.all(np.array(i_minus_j) > 0):
                print(f"\t[Player:{self.id}] Strategy: {s_i} dominates {s_j}")
                dominated_strategies.append((s_i, s_j))
            elif np.all(np.array(i_minus_j) < 0):
                print(f"\t[Player:{self.id}] Strategy: {s_j} dominates {s_i}")
                dominated_strategies.append((s_j, s_i))

        return dominated_strategies
    
    def remove_strategy(self, strat):
        self.strategy = self.strategy[~np.isin(self.strategy, strat)]
        self.belief = self.belief[~(self.belief == strat)]

In [4]:
class Game:
    def __init__(self):
        print("""Game Setup:
              Enter Number of Players(n):
                Finite Positive Integer
              Strategies for each player (Array of strings without spaces):
                ['up', 'down', 'left', 'right']
              Provide Payoffs for all strategy_profiles(s1, s2, ... sn):
                n space separated real numbers
            """)
        self.players = []
        self.strategy_sets = pd.DataFrame({})
        self.utilities = pd.DataFrame({})

    # Manually input payoffs and strategies
    def setup(self):
        n = int(input("Enter number of players: "))

        for i in range(n):
            player_strategies = input(f"Enter Strategies for Player {i}: ").lower().split(" ")
            self.players.append(Player(id = i, strat = player_strategies))
        
        self.strategy_sets = build_strategy_set(self.players) # Build strategy set for All Players
        self.setup_players()

        print("Enter Payoff for each strategy Profile:")
        payoffs = []
        for _, strategy_profile in self.strategy_sets.iterrows():
            payoff = input_payoff(f"{strategy_profile}: ", n)
            payoffs.append(payoff)
        
        self.utilities = pd.DataFrame(payoffs, columns=[f"payoff_{col}" for col in self.strategy_sets.columns])
        self.utilities = pd.concat([self.strategy_sets, self.utilities], axis = 1)

    # Load strategies and payoffs from csv file
    def load(self, path):
        self.utilities = pd.read_csv(path)
        players = [c for c in self.utilities.columns if c.isnumeric()]
        for player_id in players:
            strategies = self.utilities.loc[:, player_id].unique()
            self.players.append(Player(id = player_id, strat = strategies))

        self.setup_players()

    def setup_players(self):
        self.strategy_sets = build_strategy_set(self.players) # Build strategy set for All Players

        for index, pl in enumerate(self.players):
            pl.strategy_set_excluded = build_strategy_set(self.players[:index] + self.players[index+1:])
            # pl.build_correlated_beliefs() or pl.build_independent_beliefs()
            # TODO: We dont want to update beliefs at each iteration of deletions

    def remove_strategy(self, strat, player_id):
        self.utilities = self.utilities[~(self.utilities[player_id] == strat)]

    def view(self):
        print(f"{len(self.players)} Players")
        print(self.utilities)

In [5]:
def iesds(game):
    utilities = game.utilities
    players = game.players
    strategies_deleted = 0

    for pl in players:
        dominated_strategies = pl.find_dominated_strategies(utilities)
        if len(dominated_strategies) > 0:
            _, b = dominated_strategies[0] # b is strictly dominated, player will never play b
            print(f"Removing strategy {b} for player {pl.id}")
            pl.remove_strategy(b)
            game.remove_strategy(b, pl.id)
            game.setup_players()
            strategies_deleted += 1

    game.view()
    if strategies_deleted > 0:
        return iesds(game)
    else:
        return game

def pure_nash_equilibrium(game):
    """
    We do a brute force check on each strategy and see if unilateral deviation is feasible
    on this strategy profile. If for any player, any strategy deviation is feasible, not a NE
    """
    utilities = game.utilities
    strategy_sets = game.strategy_sets
    players = game.players
    print(f"\n{'*' * 10} Pure Strategy Nash Equilibria {'*' * 10}\n")

    for _, strategy_profile in strategy_sets.iterrows(): # For each strategy profile

        for pl in players:  # Fix pl's strategy
            s_minus_i = strategy_profile.drop(pl.id)
            s_i = strategy_profile[pl.id]
            pl_curr_util = utility(s_i, s_minus_i, pl.id, utilities)
            deviation = False

            for s_i_deviated in pl.strategy[~np.isin(pl.strategy, s_i)]: # pl's strategy deviated
                pl_deviated_util = utility(s_i_deviated, s_minus_i, pl.id, utilities)
                if pl_deviated_util > pl_curr_util: # Deviation occurs, not a NE
                    deviation = True
                    break

            if deviation == False:
                print(strategy_profile, f"\nPayoff: {pl_curr_util}\n")
                break

In [6]:
game = Game()
game.load("./csv/game1.csv")
game.view()
iesds(game)
pure_nash_equilibrium(game)

Game Setup:
              Enter Number of Players(n):
                Finite Positive Integer
              Strategies for each player (Array of strings without spaces):
                ['up', 'down', 'left', 'right']
              Provide Payoffs for all strategy_profiles(s1, s2, ... sn):
                n space separated real numbers
            
2 Players
   0  1  payoff_0  payoff_1
0  t  l         4         2
1  m  l         1         3
2  b  l         3         3
3  t  c         6         1
4  m  c         5         5
5  b  c         7         2
6  t  r        10        10
7  m  r         9         2
8  b  r         8         8
	[Player:0] Strategy: t dominates m
Removing strategy m for player 0
	[Player:1] Strategy: l dominates c
	[Player:1] Strategy: r dominates l
	[Player:1] Strategy: r dominates c
Removing strategy c for player 1
2 Players
   0  1  payoff_0  payoff_1
0  t  l         4         2
2  b  l         3         3
6  t  r        10        10
8  b  r         8         8

In [8]:
game = Game()
game.load("./csv/game2.csv")
game.view()
iesds(game)
pure_nash_equilibrium(game)

Game Setup:
              Enter Number of Players(n):
                Finite Positive Integer
              Strategies for each player (Array of strings without spaces):
                ['up', 'down', 'left', 'right']
              Provide Payoffs for all strategy_profiles(s1, s2, ... sn):
                n space separated real numbers
            
3 Players
    0  1  2  payoff_0  payoff_1  payoff_2
0   t  l  x         4         2         4
1   m  l  x         1         3         5
2   b  l  x         3         3         9
3   t  c  x         6         1         7
4   m  c  x         5         5         5
5   b  c  x         7         2         1
6   t  r  x        10        10         4
7   m  r  x         9         2         7
8   b  r  x         8         8         8
9   t  l  y         4         2        11
10  m  l  y         1         3         9
11  b  l  y         3         3         9
12  t  c  y         6         1         8
13  m  c  y         5         5         5
14  b  c 